# 1 Import packages, connect Domo

In [19]:
from pydomo import Domo
import pandas as pd
import win32com.client as win32
import datetime
import requests
import json
from datetime import datetime
import psutil

In [ ]:
#Connect with DOMO dataset
#Account Danh Bui
client_id = ''
secret = ''
domo = Domo(client_id,secret,api_host='api.domo.com') # open the connection

In [ ]:
# Input
local_path = 'C:\\Users\\Danh Bui\\Downloads\\Sheet PO of Form Nâng stock_ xóa HU_ Tạo PO_ in Label.xlsx'
sheet_name = 'Sheet1'
xlsm_path = 'O:\\Shared drives\\VEMP Supply Planning\\Production Planning\\1. Weekly Working File- Planning\\PO creation (PPDS) 202005.xlsm'
domo_new_id = ''
domo_archive_id = ''
webhook_url = ''

# 2 Read Domo data and local files

## df_new

In [22]:
ds_new_id = domo_new_id
df_new = domo.ds_get(ds_new_id)
#df_new = df_new[df_new['Timestamp'] > '2023-07-20']
df_new.tail(1)

,Mail hoàn thành,Thời \ngian \nhoàn\n thành,Status,Timestamp,Email Address,Vấn đề cần,Công thức ghép C* sau khi submit,"Số lượng (Chỉ nhập số, không nhập chữ)",BIN IM,BIN TE,...,Score,Số phút hoàn thành tạo PO,Bundle,DATE,WEEK,MONTH,Vị trí,KV2,Lý do 2,Auto
10288,NaN,NaN,NaN,2023-08-04 09:04:20,can_tran_van@colpal.com,Tạo PO,C0000009500,3400,IM1DEC01,NaN,...,0.0,NaN,HDL_K6+_OCEAN_21,45142.378009895838,NaN,8,NaN,NaN,Het PO,NaN


In [23]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10289 entries, 0 to 10288
Data columns (total 33 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   Mail hoàn thành                       9575 non-null   object        
 1   Thời \ngian \nhoàn\n thành              9649 non-null   object        
 2   Status                                  10256 non-null  object        
 3   Timestamp                               10289 non-null  datetime64[ns]
 4   Email Address                           10289 non-null  object        
 5   Vấn đề cần                              10289 non-null  object        
 6   Công thức ghép C* sau khi submit        9623 non-null   object        
 7   Số lượng (Chỉ nhập số, không nhập chữ)  9630 non-null   object        
 8   BIN IM                                  3698 non-null   object        
 9   BIN TE                                  3632 non-n

## df_old

In [24]:
df_old = pd.read_excel(local_path)
df_old.tail(1)

,Mail hoàn thành,Thời \ngian \nhoàn\n thành,Status,Timestamp,Email Address,Vấn đề cần,Công thức ghép C* sau khi submit,"Số lượng (Chỉ nhập số, không nhập chữ)",BIN IM,BIN TE,...,Score,Số phút hoàn thành tạo PO,Bundle,DATE,WEEK,MONTH,Vị trí,KV2,Lý do 2,Auto
10288,NaN,NaN,NaN,2023-08-04 09:04:20,can_tran_van@colpal.com,Tạo PO,C0000009500,3400,IM1DEC01,NaN,...,0.0,NaN,HDL_K6+_OCEAN_21,45142.378009895838,NaN,8,NaN,NaN,Het PO,NaN


In [25]:
df_old.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10289 entries, 0 to 10288
Data columns (total 33 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   Mail hoàn thành                       9575 non-null   object        
 1   Thời \ngian \nhoàn\n thành              9649 non-null   object        
 2   Status                                  10256 non-null  object        
 3   Timestamp                               10289 non-null  datetime64[ns]
 4   Email Address                           10289 non-null  object        
 5   Vấn đề cần                              10289 non-null  object        
 6   Công thức ghép C* sau khi submit        9623 non-null   object        
 7   Số lượng (Chỉ nhập số, không nhập chữ)  9630 non-null   object        
 8   BIN IM                                  3698 non-null   object        
 9   BIN TE                                  3632 non-n

# 3 Create sub datasets

In [26]:
df_new['Timestamp'].max()

Timestamp('2023-08-04 09:04:20')

In [27]:
df_old['Timestamp'].max()

Timestamp('2023-08-04 09:04:20')

In [28]:
df_pending = df_new[df_new['Timestamp'] > df_old['Timestamp'].max()]
df_pending.head(1)

,Mail hoàn thành,Thời \ngian \nhoàn\n thành,Status,Timestamp,Email Address,Vấn đề cần,Công thức ghép C* sau khi submit,"Số lượng (Chỉ nhập số, không nhập chữ)",BIN IM,BIN TE,...,Score,Số phút hoàn thành tạo PO,Bundle,DATE,WEEK,MONTH,Vị trí,KV2,Lý do 2,Auto


In [29]:
df_pending.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 0 entries
Data columns (total 33 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   Mail hoàn thành                       0 non-null      object        
 1   Thời \ngian \nhoàn\n thành              0 non-null      object        
 2   Status                                  0 non-null      object        
 3   Timestamp                               0 non-null      datetime64[ns]
 4   Email Address                           0 non-null      object        
 5   Vấn đề cần                              0 non-null      object        
 6   Công thức ghép C* sau khi submit        0 non-null      object        
 7   Số lượng (Chỉ nhập số, không nhập chữ)  0 non-null      object        
 8   BIN IM                                  0 non-null      object        
 9   BIN TE                                  0 non-null      object    

In [30]:
# Merge columns 'BIN IM', 'BIN TE', and 'BIN PKG'
def choose_not_null(row):
    if pd.notnull(row['BIN IM']):
        return row['BIN IM']
    elif pd.notnull(row['BIN TE']):
        return row['BIN TE']
    elif pd.notnull(row['BIN PKG']):
        return row['BIN PKG']
    else:
        return None

## df_pending_other 

In [31]:
df_pending_other = df_pending[['Vấn đề cần','BIN IM','BIN TE','BIN PKG','Số PO','Thông tin Batch cần tạo']]
df_pending_other = df_pending_other[df_pending_other['Vấn đề cần'] != 'Tạo PO']
df_pending_other.head(1)

,Vấn đề cần,BIN IM,BIN TE,BIN PKG,Số PO,Thông tin Batch cần tạo


In [32]:
df_pending_other['Máy'] = df_pending_other.apply(choose_not_null, axis=1)
# Drop the original columns 'BIN IM', 'BIN TE', and 'BIN PKG'
df_pending_other.drop(['BIN IM', 'BIN TE', 'BIN PKG'], axis=1, inplace=True)
# Rearrange columns order
df_pending_other = df_pending_other[['Vấn đề cần', 'Máy', 'Số PO', 'Thông tin Batch cần tạo']]
df_pending_other.head(1)

,Vấn đề cần,Máy,Số PO,Thông tin Batch cần tạo


## df_pending_create

In [33]:
df_pending_create = df_pending[df_pending['Vấn đề cần']=='Tạo PO']

In [34]:
# Filter rows with length of the string in the specified column equal to 5
# filtered_rows_len5 = df_pending_create['Công thức ghép C* sau khi submit'].str.len() == 5
filtered_rows_len5 = (df_pending_create['Công thức ghép C* sau khi submit'].str.len() == 5) & (~df_pending_create['Công thức ghép C* sau khi submit'].str.startswith('3'))

# Update the values for the filtered rows by concatenating 'C00000' to the left of the existing string
df_pending_create.loc[filtered_rows_len5, 'Công thức ghép C* sau khi submit'] = 'C00000' + df_pending_create.loc[filtered_rows_len5, 'Công thức ghép C* sau khi submit']

df_pending_create.head(1)

,Mail hoàn thành,Thời \ngian \nhoàn\n thành,Status,Timestamp,Email Address,Vấn đề cần,Công thức ghép C* sau khi submit,"Số lượng (Chỉ nhập số, không nhập chữ)",BIN IM,BIN TE,...,Score,Số phút hoàn thành tạo PO,Bundle,DATE,WEEK,MONTH,Vị trí,KV2,Lý do 2,Auto


# 4 Interact with file PO Creation xlsm

In [35]:
def get_existing_workbook(xlsm_path):
    try:
        # Try to get an instance of the running Excel application
        excel = win32.GetActiveObject("Excel.Application")
        # Check if the given file path is already open in any of the workbooks
        for workbook in excel.Workbooks:
            if workbook.FullName.lower() == file_path.lower():
                return workbook
    except:
        # If an instance of Excel is not running or the workbook is not found, return None
        return None

def open_or_get_workbook(xlsm_path):
    existing_workbook = get_existing_workbook(xlsm_path)
    if existing_workbook:
        print("Workbook is already open.")
        return existing_workbook
    else:
        excel = win32.Dispatch('Excel.Application')
        excel.Visible = True
        workbook = excel.Workbooks.Open(xlsm_path)
        return workbook

workbook = open_or_get_workbook(xlsm_path)

# Get the active worksheet named 'PO creation'
worksheet = workbook.Sheets('PO creation')

In [36]:
# Clear the data
range_to_clear_1 = worksheet.Range('A2:A100')
range_to_clear_1.ClearContents()
range_to_clear_2 = worksheet.Range('C2:I100')
range_to_clear_2.ClearContents()

True

In [37]:
df_material = pd.DataFrame(df_pending_create['Công thức ghép C* sau khi submit'])
# Write the DataFrame df_material to the worksheet
start_cell = worksheet.Range('A2')
end_cell = start_cell.Offset(len(df_material), len(df_material.columns))
df_range = worksheet.Range(start_cell, end_cell)
df_range.Value = df_material.values

In [38]:
df_info = pd.DataFrame(df_pending_create[['Số lượng (Chỉ nhập số, không nhập chữ)', 'BIN IM', 'BIN TE', 'BIN PKG', 'Ca cần tạo PO', 'Ngày cần tạo PO', 'Lý do tạo']])
df_info.head(1)

,"Số lượng (Chỉ nhập số, không nhập chữ)",BIN IM,BIN TE,BIN PKG,Ca cần tạo PO,Ngày cần tạo PO,Lý do tạo


In [39]:
df_info['Máy'] = df_info.apply(choose_not_null, axis=1)
# Drop the original columns 'BIN IM', 'BIN TE', and 'BIN PKG'
df_info.drop(['BIN IM', 'BIN TE', 'BIN PKG'], axis=1, inplace=True)
# Rearrange columns order
df_info = df_info[['Số lượng (Chỉ nhập số, không nhập chữ)', 'Máy', 'Ca cần tạo PO', 'Ngày cần tạo PO', 'Lý do tạo']]
# Format column Ca, Ngày
df_info['Ca cần tạo PO'] = df_info['Ca cần tạo PO'].astype(int)
df_info['Ngày cần tạo PO'] = df_info['Ngày cần tạo PO'].dt.strftime('%Y-%m-%d %H:%M:%S')

In [40]:
df_info.head(1)

,"Số lượng (Chỉ nhập số, không nhập chữ)",Máy,Ca cần tạo PO,Ngày cần tạo PO,Lý do tạo


In [41]:
# Write the DataFrame df_info to the worksheet
start_cell = worksheet.Range('C2')
end_cell = start_cell.Offset(len(df_info), len(df_info.columns))
df_range = worksheet.Range(start_cell, end_cell)
df_range.Value = df_info.values

In [42]:
try:    
    # Run the scirpt PO Creation
    workbook.Application.Run('PO_CREATION')
except:
    print("Error occurred")

Error occurred


# 5 Send message to Google Chat

In [43]:
first_row = 2
last_row = len(df_material) + 1

In [44]:
last_row

1

In [45]:
C = [worksheet.Cells(i, 1).Value for i in range(first_row, last_row +1)]
Qty = [worksheet.Cells(i, 3).Value for i in range(first_row, last_row +1)]
Shift = [worksheet.Cells(i, 5).Value for i in range(first_row, last_row +1)]

Date = [worksheet.Cells(i, 6).Value for i in range(first_row, last_row +1)]
# Convert ISO 8601 date values to string with format 'm/d/Y'
Date = [datetime.strptime(str(d)[:10], "%Y-%m-%d").strftime("%m/%d/%Y") for d in Date]

Stt = [worksheet.Cells(i, 8).Value for i in range(first_row, last_row +1)]
# Replace None values with "PO Creation Failed"
Stt = ["PO Creation Failed" if value is None else value for value in Stt]

PO = [worksheet.Cells(i, 9).Value for i in range(first_row, last_row +1)]
# Replace None values with "PO Creation Failed"
PO = ["N/A" if value is None else value for value in PO]

Machine = [worksheet.Cells(i, 4).Value for i in range(first_row, last_row +1)]

In [46]:
# Create a DataFrame from the data array with column name "PO Creation"
df_ggchat_pending_po = pd.DataFrame({'Material': C,
                                     'Quantity' : Qty,
                                     'Shift': Shift,
                                     'Date' : Date,
                                     'Status': Stt, 
                                     'PO number': PO,
                                    'Machine': Machine})

In [47]:
df_ggchat_pending_po['Quantity'] = df_ggchat_pending_po['Quantity'].astype(float).astype(int)
df_ggchat_pending_po['Shift'] = df_ggchat_pending_po['Shift'].astype(float).astype(int)
df_ggchat_pending_po.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Material   0 non-null      float64
 1   Quantity   0 non-null      int32  
 2   Shift      0 non-null      int32  
 3   Date       0 non-null      float64
 4   Status     0 non-null      float64
 5   PO number  0 non-null      float64
 6   Machine    0 non-null      float64
dtypes: float64(5), int32(2)
memory usage: 124.0 bytes


In [48]:
df_ggchat_pending_po.head(1)

,Material,Quantity,Shift,Date,Status,PO number,Machine


In [49]:
tab = "\t"

## Success PO

In [50]:
df_ggchat_success = df_ggchat_pending_po[df_ggchat_pending_po['Status'] != 'PO Creation Failed']
df_ggchat_success = df_ggchat_success[['Material','Status','PO number']]
df_ggchat_success.head(1)

,Material,Status,PO number


In [51]:
if len(df_ggchat_success) > 0:
    # Create the header text in bold format using Markdown formatting
    title = "\r *SUCCESSFULLY CREATED PO* \r"
    message = ""

    # Iterate through the DataFrame to send individual messages for each row
    for index, row in df_ggchat_success.iterrows():
        message = message + "Material: " + str(row['Material']) + tab + "Status: " + str(row['Status']) + tab + "PO number: " + str(row['PO number']) + "\r"

    message = title + message

    # Convert the message dictionary to JSON format
    json_data = json.dumps({"text": message})

    # Send the POST request with the JSON data to the webhook URL
    response = requests.post(webhook_url, data=json_data, headers={"Content-Type": "application/json"})

## Fail PO

In [52]:
df_ggchat_fail = df_ggchat_pending_po[df_ggchat_pending_po['Status'] == 'PO Creation Failed']
df_ggchat_fail = df_ggchat_fail[['Material','Quantity','Shift','Date']]
df_ggchat_fail.head(1)

,Material,Quantity,Shift,Date


In [53]:
# if len(df_ggchat_fail) > 0:
#     # Create the header text in bold format using Markdown formatting
#     title = "\r *FAILED TO CREATE PO* \r"
#     header = "Material " + tab  + tab + "Quantity " + tab + "Shift " + tab + "Date " + "\r"
#     message = ""

#     # Iterate through the DataFrame to send individual messages for each row
#     for index, row in df_ggchat_fail.iterrows():
#         message = message + str(row['Material']) + tab + str(row['Quantity']) + tab + str(row['Shift']) + tab + str(row['Date']) + "\r"

#     message = title + header + message

#     # Convert the message dictionary to JSON format
#     json_data = json.dumps({"text": message})

#     # Send the POST request with the JSON data to the webhook URL
#     response = requests.post(webhook_url, data=json_data, headers={"Content-Type": "application/json"})

In [54]:
if len(df_ggchat_fail) > 0:
    # Create the header text in bold format using Markdown formatting
    title = "\r *FAILED TO CREATE PO* \r"
    message = ""

    # Iterate through the DataFrame to send individual messages for each row
    for index, row in df_ggchat_fail.iterrows():
        message = message + "Material: " + str(row['Material']) + tab + "Quantity: " + str(row['Quantity']) + tab + "Shift: " + str(row['Shift']) + tab + "Date: " + str(row['Date']) + "\r"

    message = title + message

    # Convert the message dictionary to JSON format
    json_data = json.dumps({"text": message})

    # Send the POST request with the JSON data to the webhook URL
    response = requests.post(webhook_url, data=json_data, headers={"Content-Type": "application/json"})

## Other cases

In [55]:
df_ggchat_other = df_pending_other

In [56]:
# Rename the columns in the DataFrame
df_ggchat_other = df_ggchat_other.rename(columns={
    'Vấn đề cần': 'Problem',
    'Máy': 'Machine',
    'Số PO': 'PO',
    'Thông tin Batch cần tạo': 'Info'
})

In [57]:
df_ggchat_other.head(1)

,Problem,Machine,PO,Info


In [58]:
if len(df_ggchat_other) > 0:
    # Create the header text in bold format using Markdown formatting
    title = "\r *OTHER PENDING* \r"
    message = ""

    # Iterate through the DataFrame to send individual messages for each row
    for index, row in df_ggchat_other.iterrows():
        message = message + "Problem: " + str(row['Problem']) + tab + "Machine: " + str(row['Machine']) + tab + "PO: " + str(row['PO']) + tab + "Info: " + str(row['Info']) + "\r"

    message = title + message

    # Convert the message dictionary to JSON format
    json_data = json.dumps({"text": message})

    # Send the POST request with the JSON data to the webhook URL
    response = requests.post(webhook_url, data=json_data, headers={"Content-Type": "application/json"})

# 6 Archive data

## Update local excel file

In [59]:
len(df_pending)

0

In [60]:
if len(df_pending) > 0:
    # Write the DataFrame to the Excel file
    df_new.to_excel(local_path, sheet_name=sheet_name, index=False)

## Append result to Domo

In [61]:
# Retrieve df_pending and merge with df_ggchat_pending_po to get status and push to Domo

In [62]:
df_pending_temp = df_pending[['Timestamp','Email Address','Vấn đề cần','Công thức ghép C* sau khi submit','Số lượng (Chỉ nhập số, không nhập chữ)','BIN IM','BIN TE','BIN PKG','Ca cần tạo PO','Ngày cần tạo PO','Lý do tạo','Planner/ OE Note Lý do tạo đúng','Số PO','Thông tin Batch cần tạo']].copy()
df_pending_temp['Máy'] = df_pending_temp.apply(choose_not_null, axis=1)
# Drop the original columns 'BIN IM', 'BIN TE', and 'BIN PKG'
df_pending_temp.drop(['BIN IM', 'BIN TE', 'BIN PKG'], axis=1, inplace=True)
df_pending_temp['Số lượng (Chỉ nhập số, không nhập chữ)'] = df_pending_temp['Số lượng (Chỉ nhập số, không nhập chữ)'].astype(float).astype(int)
df_pending_temp['Ca cần tạo PO'] = df_pending_temp['Ca cần tạo PO'].astype(float).astype(int)
df_pending_temp['Ngày cần tạo PO'] = df_pending_temp['Ngày cần tạo PO'].dt.strftime('%m/%d/%Y')

# Filter rows with length of the string in the specified column equal to 5
filtered_rows_len5_2 = df_pending_temp['Công thức ghép C* sau khi submit'].str.len() == 5
# Update the values for the filtered rows by concatenating 'C00000' to the left of the existing string
df_pending_temp.loc[filtered_rows_len5_2, 'Công thức ghép C* sau khi submit'] = 'C00000' + df_pending_temp.loc[filtered_rows_len5_2, 'Công thức ghép C* sau khi submit']

df_pending_temp.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 0 entries
Data columns (total 12 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   Timestamp                               0 non-null      datetime64[ns]
 1   Email Address                           0 non-null      object        
 2   Vấn đề cần                              0 non-null      object        
 3   Công thức ghép C* sau khi submit        0 non-null      object        
 4   Số lượng (Chỉ nhập số, không nhập chữ)  0 non-null      int32         
 5   Ca cần tạo PO                           0 non-null      int32         
 6   Ngày cần tạo PO                         0 non-null      object        
 7   Lý do tạo                               0 non-null      object        
 8   Planner/ OE Note Lý do tạo đúng         0 non-null      object        
 9   Số PO                                   0 non-null      object    

In [63]:
df_ggchat_pending_po_temp = df_ggchat_pending_po[['Material','Quantity','Shift','Date','Status','Machine']].copy()

# Create a dictionary to map the old column names to the new column names
column_rename_map_2 = {
    'Material': 'Công thức ghép C* sau khi submit',
    'Quantity': 'Số lượng (Chỉ nhập số, không nhập chữ)',
    'Shift': 'Ca cần tạo PO',
    'Date': 'Ngày cần tạo PO',
    'Status': 'Trạng thái',
    'Machine': 'Máy'
}

# Rename the columns using the rename() method
df_ggchat_pending_po_temp.rename(columns=column_rename_map_2, inplace=True)

df_ggchat_pending_po_temp['Công thức ghép C* sau khi submit'] = df_ggchat_pending_po_temp['Công thức ghép C* sau khi submit'].astype(str)
df_ggchat_pending_po_temp['Ngày cần tạo PO'] = df_ggchat_pending_po_temp['Ngày cần tạo PO'].astype(str)

df_ggchat_pending_po_temp.head(1)

,Công thức ghép C* sau khi submit,"Số lượng (Chỉ nhập số, không nhập chữ)",Ca cần tạo PO,Ngày cần tạo PO,Trạng thái,Máy


In [64]:
df_pending_temp.head(1)

,Timestamp,Email Address,Vấn đề cần,Công thức ghép C* sau khi submit,"Số lượng (Chỉ nhập số, không nhập chữ)",Ca cần tạo PO,Ngày cần tạo PO,Lý do tạo,Planner/ OE Note Lý do tạo đúng,Số PO,Thông tin Batch cần tạo,Máy


In [65]:
df_ggchat_pending_po_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 6 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Công thức ghép C* sau khi submit        0 non-null      object 
 1   Số lượng (Chỉ nhập số, không nhập chữ)  0 non-null      int32  
 2   Ca cần tạo PO                           0 non-null      int32  
 3   Ngày cần tạo PO                         0 non-null      object 
 4   Trạng thái                              0 non-null      float64
 5   Máy                                     0 non-null      float64
dtypes: float64(2), int32(2), object(2)
memory usage: 124.0+ bytes


In [66]:
df_pending_temp.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 0 entries
Data columns (total 12 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   Timestamp                               0 non-null      datetime64[ns]
 1   Email Address                           0 non-null      object        
 2   Vấn đề cần                              0 non-null      object        
 3   Công thức ghép C* sau khi submit        0 non-null      object        
 4   Số lượng (Chỉ nhập số, không nhập chữ)  0 non-null      int32         
 5   Ca cần tạo PO                           0 non-null      int32         
 6   Ngày cần tạo PO                         0 non-null      object        
 7   Lý do tạo                               0 non-null      object        
 8   Planner/ OE Note Lý do tạo đúng         0 non-null      object        
 9   Số PO                                   0 non-null      object    

In [67]:
domo_archive_data = df_pending_temp.merge(df_ggchat_pending_po_temp, how='left', 
                                     on=['Công thức ghép C* sau khi submit','Số lượng (Chỉ nhập số, không nhập chữ)','Ca cần tạo PO','Ngày cần tạo PO','Máy'])
domo_archive_data.head(1)

,Timestamp,Email Address,Vấn đề cần,Lý do tạo,Planner/ OE Note Lý do tạo đúng,Số PO,Thông tin Batch cần tạo,Công thức ghép C* sau khi submit,"Số lượng (Chỉ nhập số, không nhập chữ)",Ca cần tạo PO,Ngày cần tạo PO,Trạng thái,Máy


In [68]:
# Convert the current timestamp to the desired string format
update_at_str = datetime.now().strftime('%Y/%m/%d %H:%M:%S')
# Add the "Update at" column to the DataFrame
domo_archive_data['Update at'] = update_at_str

In [69]:
datetime.now().strftime('%Y/%m/%d %H:%M:%S')

'2023/08/04 11:19:36'

In [70]:
# Define the desired column order
desired_column_order = ['Update at', 'Trạng thái', 'Timestamp', 'Email Address', 'Vấn đề cần', 'Công thức ghép C* sau khi submit',
                        'Số lượng (Chỉ nhập số, không nhập chữ)', 'Máy', 'Ca cần tạo PO', 'Ngày cần tạo PO',
                        'Lý do tạo', 'Planner/ OE Note Lý do tạo đúng', 'Số PO', 'Thông tin Batch cần tạo']

# Reorder the columns in the DataFrame
domo_archive_data = domo_archive_data.reindex(columns=desired_column_order)

In [71]:
domo_archive_data.head(1)

,Update at,Trạng thái,Timestamp,Email Address,Vấn đề cần,Công thức ghép C* sau khi submit,"Số lượng (Chỉ nhập số, không nhập chữ)",Máy,Ca cần tạo PO,Ngày cần tạo PO,Lý do tạo,Planner/ OE Note Lý do tạo đúng,Số PO,Thông tin Batch cần tạo


In [72]:
if len(domo_archive_data) > 0:
    # Read the existing data from the Domo dataset
    existing_data = domo.ds_get(domo_archive_id)

    # Combine the existing data with the new data (df_pending)
    combined_data = pd.concat([existing_data, domo_archive_data], ignore_index=True)

    # Update the Domo dataset with the combined data
    domo.ds_update(domo_archive_id, combined_data)

## Save and close xlsm file

In [73]:
 # Save and close the workbook
workbook.Save()
workbook.Close()
print('DONE')

DONE
